# RFM Analysis

## What is RFM analysis?

**RFM** stands for **Recency**, **Frequency**, and **Monetary value**. It is a customer segmentation method that uses three behavioral metrics:

- **Recency**: How recently did the customer last purchase? (e.g. number of days since their most recent order.)  
  *More recent = typically more engaged.*

- **Frequency**: How often do they buy? (e.g. number of orders or transactions in the analysis period.)  
  *More orders = habitually active.*

- **Monetary**: How much do they spend? (e.g. total revenue from that customer.)  
  *Higher spend = more valuable to the business.*

Customers are scored on each dimension (e.g. 1–5, with 5 = best). Those scores are then combined to assign segments—for example "Champions" (high on all three) or "At Risk" (high Frequency and Monetary but low Recency).

---

## How does it add value to businesses?

- **Targeted marketing**: Send different messages and offers by segment—win-back emails for lapsed high-value customers, loyalty rewards for top customers, and activation campaigns for one-time buyers.
- **Resource allocation**: Focus retention efforts and premium support on high-value segments; reduce spend on long-inactive, low-value segments.
- **Product and pricing**: See which segments drive revenue and what they buy; test promotions or pricing by segment.
- **Churn and opportunity**: Low Recency with high historical Frequency/Monetary highlights at-risk customers to reactivate; high Recency with low Frequency highlights room to increase repeat purchase.

---

## What can we learn from RFM on an e-commerce dataset like ours?

- **Who are our best customers?** (high Recency, Frequency, and Monetary)—how many there are and how much revenue they represent.
- **Who has lapsed but used to be valuable?** (low Recency, high Frequency/Monetary)—priority for win-back campaigns.
- **Who are recent but infrequent or low-spend?** (high Recency, low Frequency or Monetary)—opportunity to convert or upsell.
- **Who are low-value and long-inactive?** (low on all three)—candidates to deprioritise or exclude from costly campaigns.
- **Segment mix**: Whether revenue is concentrated in a few "Champion" customers or spread across many segments—with implications for risk and strategy.

In [ ]:
# Load data and apply same cleaning as main retail analysis
import pandas as pd

file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["CustomerID", "Description"])
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

print("Cleaned shape:", df_clean.shape)
print("Date range:", df_clean["InvoiceDate"].min(), "to", df_clean["InvoiceDate"].max())

Cleaned shape: (392732, 9)
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [2]:
# Compute Recency, Frequency, Monetary per customer
reference_date = df_clean["InvoiceDate"].max()

rfm = (
    df_clean
    .groupby("CustomerID")
    .agg(
        last_order_date=("InvoiceDate", "max"),
        order_count=("InvoiceNo", "nunique"),
        total_revenue=("Revenue", "sum")
    )
    .reset_index()
)

rfm["Recency"] = (reference_date - rfm["last_order_date"]).dt.days
rfm["Frequency"] = rfm["order_count"]
rfm["Monetary"] = rfm["total_revenue"]

rfm = rfm[["CustomerID", "Recency", "Frequency", "Monetary"]]

print("RFM stats (first 5 rows):")
print(rfm.head())
print("\nR/F/M summary:")
print(rfm[["Recency", "Frequency", "Monetary"]].describe())

RFM stats (first 5 rows):
   CustomerID  Recency  Frequency  Monetary
0       12346      325          1  77183.60
1       12347        1          7   4310.00
2       12348       74          4   1797.24
3       12349       18          1   1757.55
4       12350      309          1    334.40

R/F/M summary:
           Recency    Frequency       Monetary
count  4339.000000  4339.000000    4339.000000
mean     91.518322     4.271952    2048.215924
std     100.009747     7.705493    8984.248352
min       0.000000     1.000000       0.000000
25%      17.000000     1.000000     306.455000
50%      50.000000     2.000000     668.560000
75%     141.000000     5.000000    1660.315000
max     373.000000   210.000000  280206.020000


In [7]:
# Score R, F, M into quintiles (1-5). Recency: lower days = better = higher score.
rfm["R_score"] = pd.qcut(rfm["Recency"].rank(method="first"), q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_Score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)

print("Score distributions:")
print(rfm[["R_score", "F_score", "M_score"]].value_counts().sort_index())
print("\nSample with scores:")
print(rfm.head(10))

Score distributions:
R_score  F_score  M_score
1        1        1          183
                  2          127
                  3           33
                  4           18
                  5            4
                            ... 
5        5        1            1
                  2            1
                  3            8
                  4           82
                  5          347
Name: count, Length: 118, dtype: int64

Sample with scores:
   CustomerID  Recency  Frequency  Monetary  R_score  F_score  M_score  \
0       12346      325          1  77183.60        1        1        5   
1       12347        1          7   4310.00        5        5        5   
2       12348       74          4   1797.24        2        4        4   
3       12349       18          1   1757.55        4        1        4   
4       12350      309          1    334.40        1        1        2   
5       12352       35          8   2506.04        3        5        5   
6       1235

In [4]:
# Assign named segments from R, F, M scores
def rfm_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 4 and f >= 2 and m >= 2:
        return "Loyal Customers"
    if r >= 4 and f <= 2 and m <= 2:
        return "New Customers"
    if r >= 3 and f >= 3 and m >= 3:
        return "Potential Loyalists"
    if r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    if r <= 2 and f >= 2 and m >= 2:
        return "Hibernating"
    if r <= 2 and f <= 2 and m <= 2:
        return "Lost"
    if r >= 3 and f <= 2 and m <= 2:
        return "Promising"
    return "Need Attention"

rfm["Segment"] = rfm.apply(rfm_segment, axis=1)

print("Segment counts:")
print(rfm["Segment"].value_counts().sort_values(ascending=False))
print("\nSegment summary (customers and total revenue):")
segment_summary = (
    rfm
    .groupby("Segment", as_index=False)
    .agg(customers=("CustomerID", "count"), total_revenue=("Monetary", "sum"))
    .sort_values("total_revenue", ascending=False)
)
print(segment_summary.to_string(index=False))

Segment counts:
Segment
Champions              943
Lost                   673
Loyal Customers        562
At Risk                465
Potential Loyalists    433
Hibernating            430
Need Attention         410
Promising              251
New Customers          172
Name: count, dtype: int64

Segment summary (customers and total revenue):
            Segment  customers  total_revenue
          Champions        943    5738885.380
Potential Loyalists        433     949932.411
            At Risk        465     763252.201
    Loyal Customers        562     600323.260
     Need Attention        410     349316.850
        Hibernating        430     251252.392
               Lost        673     135832.420
          Promising        251      63157.720
      New Customers        172      35256.260
